# Обучение моделей (МРТ, опухоли головного мозга)

Ноутбук только запускает код проекта: вся логика — в `src/`. Подробная инструкция — `docs/kaggle.md` в проекте.

**Перед запуском** (панель справа):
1. *Input* → подключены два набора данных: с кодом (`code_bundle.zip`) и с данными (`data_bundle.zip`).
2. *Settings → Accelerator* → **GPU T4**.
3. *Settings → Internet* → **On** (нужен для установки пакетов и предобученных весов).

Запуски задаются в первой ячейке: `конфиг:seed` через пробел.

In [ ]:
# Запуски «конфиг:seed» через пробел. По умолчанию — финальные модели: классификатор и сегментатор, seed 42.
RUNS = "run1_cls:42 run1_seg_neg:42"
EXTRA = ""                          # дополнительные аргументы src.train; обычно пусто

import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
CODE = WORK / "code"

# Код берется из набора данных с code_bundle.zip. Kaggle обычно распаковывает архив сам; если нет — распаковываем здесь.
shutil.rmtree(CODE, ignore_errors=True)
unpacked, packed = sorted(INPUT.rglob("kaggle_run.py")), sorted(INPUT.rglob("code_bundle.zip"))
if unpacked:
    root = unpacked[-1].parents[1]
    CODE.mkdir(parents=True)
    for name in ("src", "configs"):
        shutil.copytree(root / name, CODE / name)
    for name in ("COMMIT", "requirements-kaggle.txt"):
        shutil.copy(root / name, CODE / name)
elif packed:
    with zipfile.ZipFile(packed[-1]) as z:
        z.extractall(CODE)
else:
    raise SystemExit("код не найден: подключите набор данных с code_bundle.zip (Add Input)")
print("код:", CODE, "| коммит", (CODE / "COMMIT").read_text().strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE / "requirements-kaggle.txt")], check=True)
print("пакеты установлены")

In [ ]:
!cd {CODE} && {sys.executable} -m src.kaggle_run --runs {RUNS} --extra "{EXTRA}"

### Результаты

В `/kaggle/working` лежит архив `results_<время>.zip`: история обучения, предсказания на val, веса и `experiments.csv`.
Скачайте его (панель справа → *Output*) и внесите в проект: `python -m src.import_kaggle <файл>`.

Длинные запуски удобнее выполнять через *Save Version → Save & Run All (Commit)*: вкладку можно закрыть, архив появится на странице ноутбука в разделе *Output*.